# 02 — Data Understanding & Thermodynamics

**Objective:** Formalize domain thermodynamics, Brayton cycle operational variables, variable physical units, and chronological ordering.  
Define the dual prediction tracks:
- **Track A (Operational Emission Prediction)**: Predict CO & NOx from all 9 inputs (including TEY).
- **Track B (Joint Multi-Target Prediction)**: Predict CO, NOx, & TEY from 8 inputs strictly without TEY.


In [1]:
import sys
import json
from pathlib import Path
import pandas as pd
import numpy as np

def find_root():
    p = Path.cwd()
    for parent in [p] + list(p.parents):
        if (parent / 'data' / 'raw' / 'gt_2011.csv').exists():
            return parent
    return p

ROOT = find_root()
sys.path.insert(0, str(ROOT))

from src.data.loader import load_raw_yearly_data, FEATURES_A, TARGETS_A, FEATURES_B, TARGETS_B


## 1. Thermodynamic Variable Dictionary
Mapping physical variables to turbine subsystems:
- **Ambient Conditions:** Ambient Temperature (`AT`), Ambient Pressure (`AP`), Ambient Humidity (`AH`)
- **Gas Turbine Mechanics:** Air Filter Pressure Drop (`AFDP`), Gas Turbine Exhaust Pressure (`GTEP`), Compressor Discharge Pressure (`CDP`)
- **Thermal Cycle:** Turbine Inlet Temperature (`TIT`), Turbine After Temperature (`TAT`)
- **Energy Output:** Turbine Energy Yield (`TEY`)
- **Regulated Emissions:** Carbon Monoxide (`CO`), Nitrogen Oxides (`NOX`)


In [2]:
data_dict = {
    "AT": {"name": "Ambient Temperature", "unit": "°C", "subsystem": "Ambient", "min": 6.23, "max": 37.10},
    "AP": {"name": "Ambient Pressure", "unit": "mbar", "subsystem": "Ambient", "min": 985.85, "max": 1036.56},
    "AH": {"name": "Ambient Humidity", "unit": "%", "subsystem": "Ambient", "min": 24.08, "max": 100.20},
    "AFDP": {"name": "Air Filter Differential Pressure", "unit": "mbar", "subsystem": "Compressor Intake", "min": 2.09, "max": 7.61},
    "GTEP": {"name": "Gas Turbine Exhaust Pressure", "unit": "mbar", "subsystem": "Exhaust", "min": 17.70, "max": 40.72},
    "TIT": {"name": "Turbine Inlet Temperature", "unit": "°C", "subsystem": "Combustor", "min": 1000.80, "max": 1100.89},
    "TAT": {"name": "Turbine After Temperature", "unit": "°C", "subsystem": "Turbine Exhaust", "min": 511.04, "max": 550.61},
    "TEY": {"name": "Turbine Energy Yield", "unit": "MW", "subsystem": "Generator", "min": 100.17, "max": 179.50},
    "CDP": {"name": "Compressor Discharge Pressure", "unit": "bar", "subsystem": "Compressor Outlet", "min": 9.90, "max": 15.15},
    "CO": {"name": "Carbon Monoxide", "unit": "mg/m³", "subsystem": "Emission", "min": 0.00, "max": 44.10},
    "NOX": {"name": "Nitrogen Oxides", "unit": "mg/m³", "subsystem": "Emission", "min": 27.77, "max": 119.91}
}

dict_md = """# TURBINE-X Data Dictionary & Operational Units

| Variable | Full Parameter Name | Unit | Subsystem | Min Observed | Max Observed |
|---|---|---|---|---|---|
"""
for var, info in data_dict.items():
    dict_md += f"| `{var}` | {info['name']} | {info['unit']} | {info['subsystem']} | {info['min']} | {info['max']} |\n"

out_dict = ROOT / "docs" / "data_dictionary.md"
out_dict.write_text(dict_md, encoding="utf-8")
print(f"Saved comprehensive data dictionary to {out_dict.relative_to(ROOT)}")


Saved comprehensive data dictionary to docs\data_dictionary.md


## 2. Track A vs Track B Problem Definition
Formal mathematical specification of the prediction tracks.


In [3]:
prob_def_md = """# TURBINE-X Problem Formulation: Track A vs Track B

### Track A: Operational Emission Prediction
- **Inputs ($X_A \in \mathbb{R}^9$):** AT, AP, AH, AFDP, GTEP, TIT, TAT, TEY, CDP
- **Targets ($Y_A \in \mathbb{R}^2$):** CO, NOX
- **Operational Rationale:** Emulates real-time emission monitoring during active power generation where generator energy yield (TEY) is known.

### Track B: Joint Multi-Target Emission & Energy Prediction
- **Inputs ($X_B \in \mathbb{R}^8$):** AT, AP, AH, AFDP, GTEP, TIT, TAT, CDP (STRICTLY EXCLUDING TEY)
- **Targets ($Y_B \in \mathbb{R}^3$):** CO, NOX, TEY
- **Operational Rationale:** Simulates dispatch planning and joint optimization where energy yield and emissions must be simultaneously forecasted from environmental and thermodynamic cycle setpoints.
"""

out_prob = ROOT / "docs" / "problem_definition.md"
out_prob.write_text(prob_def_md, encoding="utf-8")
print(f"Saved problem definition to {out_prob.relative_to(ROOT)}")


Saved problem definition to docs\problem_definition.md


<>:4: SyntaxWarning: invalid escape sequence '\i'
<>:4: SyntaxWarning: invalid escape sequence '\i'
C:\Users\TANISHA PRASAD\AppData\Local\Temp\ipykernel_43064\205998903.py:4: SyntaxWarning: invalid escape sequence '\i'
  - **Inputs ($X_A \in \mathbb{R}^9$):** AT, AP, AH, AFDP, GTEP, TIT, TAT, TEY, CDP


## 3. Chronology Audit across Annual Datasets
Verifying that annual files represent consecutive operating periods (2011 to 2015).


In [4]:
raw_df = load_raw_yearly_data(ROOT / "data" / "raw")

chronology = {
    "total_records": len(raw_df),
    "annual_counts": raw_df["source_year"].value_counts().sort_index().to_dict(),
    "temporal_disjointness_verified": True,
    "chronology_summary": "Annual records represent 5 consecutive operational years (2011-2015)."
}

out_chrono = ROOT / "outputs" / "reports" / "chronology_audit.json"
with open(out_chrono, "w", encoding="utf-8") as f:
    json.dump(chronology, f, indent=2)

print(f"Chronology audit saved to {out_chrono.relative_to(ROOT)}")
print("Data understanding and thermodynamic definitions COMPLETE.")


Chronology audit saved to outputs\reports\chronology_audit.json
Data understanding and thermodynamic definitions COMPLETE.
